# 🧠 Rock-Paper-Scissors — MobileNetV2 (Transfer Learning + Fine-Tuning)

---

## 🎯 Objective

Use **MobileNetV2** (pre-trained on ImageNet) to classify hand gesture images via a
**two-phase training strategy**:

| Phase | Strategy | What Trains | Learning Rate |
|-------|----------|-------------|---------------|
| **Phase 1** | Transfer Learning | Only the new classification head | Default (Adam) |
| **Phase 2** | Fine-Tuning | Top layers of MobileNetV2 + head | Very low (1e-5) |

### What You Will Learn

| Topic | Description |
|-------|-------------|
| Transfer learning | Reuse features learned on ImageNet for a new task |
| Fine-tuning | Unfreeze top layers and train with a tiny learning rate |
| Functional API | Build models with shared layers using `keras.Model` |
| Two-phase training | Train head first, then fine-tune the backbone |
| Callbacks | EarlyStopping, ReduceLROnPlateau, ModelCheckpoint |

### Prerequisites

- Run **`data_split.ipynb`** first to generate `split_data/`.

---

## Step 1 — Import Libraries

In [ ]:
import keras
import tensorflow as tf
import matplotlib.pyplot as plt
from keras import layers
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

---

## Step 2 — Configuration

In [ ]:
# ╔══════════════════════════════════════════════╗
# ║        CONFIGURATION — CHANGE HERE           ║
# ╚══════════════════════════════════════════════╝

IMAGE_SIZE        = (150, 150)
BATCH_SIZE        = 32
SEED              = 42

# Phase 1: Transfer Learning
PHASE1_EPOCHS     = 15
PHASE1_WEIGHTS    = '../weights/mobilenet/transferlearning'

# Phase 2: Fine-Tuning
PHASE2_EPOCHS     = 10          # additional epochs after phase 1
FINE_TUNE_AT      = 100         # unfreeze layers from this index onwards
FINE_TUNE_LR      = 1e-5
PHASE2_WEIGHTS    = '../weights/mobilenet/finetune'

# Paths
TRAIN_DIR         = '../split_data/train'
VAL_DIR           = '../split_data/val'
TEST_DIR          = '../split_data/test'

os.makedirs(PHASE1_WEIGHTS, exist_ok=True)
os.makedirs(PHASE2_WEIGHTS, exist_ok=True)

keras.utils.set_random_seed(SEED)

---

## Step 3 — Load the Data

We load from the pre-split directories — no inline `validation_split` needed.

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    seed=SEED,
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)

print(f"\n📦 Classes: {CLASS_NAMES}")
print(f"📦 Number of classes: {NUM_CLASSES}")

---

## Step 4 — Visualize Raw Samples

In [ ]:
def show_batch(dataset, class_names, title, n=9):
    plt.figure(figsize=(8, 8))
    for images, labels in dataset.take(1):
        for i in range(min(n, images.shape[0])):
            ax = plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype('uint8'))
            plt.title(class_names[int(labels[i])])
            plt.axis('off')
    plt.suptitle(title, fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

show_batch(train_ds, CLASS_NAMES, 'Training samples (raw)')
show_batch(test_ds,  CLASS_NAMES, 'Test samples (raw)')

---

## Step 5 — Optimize the Input Pipeline

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

---

## Step 6 — Data Augmentation

Same augmentations as the from-scratch CNN, applied as Keras preprocessing layers.

In [ ]:
data_augmentation = keras.Sequential([
    # ── Spatial ──
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),

    # ── Colour / Appearance ──
    layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2),

    # ── Noise / Regularization ──
    layers.GaussianNoise(0.05),
], name='data_augmentation')

### Preview Augmentations

In [ ]:
plt.figure(figsize=(8, 8))
for images, _ in train_ds.take(1):
    first = images[0]
    for i in range(9):
        augmented = data_augmentation(tf.expand_dims(first, 0), training=True)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(tf.cast(augmented[0], tf.uint8))
        plt.axis('off')
plt.suptitle('Same image, 9 different augmentations', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Step 7 — Build the Model (Functional API)

### Architecture

```
Input → Augmentation → MobileNetV2 preprocessing → MobileNetV2 (frozen)
      → GlobalAveragePooling2D → Dropout(0.2) → Dense(softmax)
```

- **MobileNetV2** is loaded with ImageNet weights but **without** the top classification layer.
- We add our own classification head for 3 classes.
- The base model is **frozen** initially — only the head trains in Phase 1.

In [ ]:
# Load MobileNetV2 backbone (frozen)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=IMAGE_SIZE + (3,),
)
base_model.trainable = False

# Build full model using Functional API
inputs = keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)
model.summary()

---

## Phase 1 — Transfer Learning

### What Happens

- The MobileNetV2 backbone is **completely frozen** — its weights don't change.
- Only the new classification head (GlobalAveragePooling + Dropout + Dense) is trained.
- This is fast because very few parameters are updated.

### Why Do This First?

If we fine-tune the entire model from the start, the randomly initialized head
would produce large gradients that **corrupt the pre-trained features**.
Training the head first ensures it produces reasonable outputs before we touch the backbone.

### Compile for Phase 1

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

### Callbacks for Phase 1

In [ ]:
phase1_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(PHASE1_WEIGHTS, 'phase1_best.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=0,
    ),
]

### Train Phase 1

In [ ]:
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=phase1_callbacks,
)

print(f"\n✅ Phase 1 complete. Best model saved to: {PHASE1_WEIGHTS}/phase1_best.keras")

### Phase 1 — Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\n📊 Phase 1 — Test Accuracy: {test_acc:.4f}")
print(f"📊 Phase 1 — Test Loss:     {test_loss:.4f}")

### Phase 1 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_phase1.history['accuracy'], label='Train Acc')
axes[0].plot(history_phase1.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Phase 1 — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_phase1.history['loss'], label='Train Loss')
axes[1].plot(history_phase1.history['val_loss'], label='Val Loss')
axes[1].set_title('Phase 1 — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('MobileNetV2 — Phase 1: Transfer Learning', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---

## Phase 2 — Fine-Tuning

### What Happens

- We **unfreeze** the top layers of MobileNetV2 (from layer 100 onwards).
- We use a **very small learning rate** (1e-5) to gently adjust the pre-trained weights.
- The bottom layers stay frozen — they capture universal features (edges, textures).

### Why Only the Top Layers?

- **Bottom layers** → generic features (edges, gradients) — useful for all images.
- **Top layers** → task-specific features (shapes, objects) — worth adjusting.

### Unfreeze Top Layers

In [ ]:
base_model.trainable = True

# Freeze all layers BEFORE `FINE_TUNE_AT`
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

# Count trainable vs frozen
trainable   = sum(1 for l in base_model.layers if l.trainable)
frozen      = sum(1 for l in base_model.layers if not l.trainable)
print(f"🔓 Trainable layers: {trainable}")
print(f"🔒 Frozen layers:    {frozen}")

### Re-compile for Phase 2

> ⚠️ **You must re-compile** after changing `trainable` attributes — the optimizer needs to
> be rebuilt with the new set of trainable parameters.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

### Callbacks for Phase 2

In [ ]:
phase2_callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(PHASE2_WEIGHTS, 'phase2_best.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=0,
    ),
]

### Train Phase 2

We continue training from where Phase 1 left off using `initial_epoch`.

In [ ]:
total_phase1_epochs = history_phase1.epoch[-1] + 1

history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_phase1_epochs + PHASE2_EPOCHS,
    initial_epoch=total_phase1_epochs,
    callbacks=phase2_callbacks,
)

print(f"\n✅ Phase 2 complete. Best model saved to: {PHASE2_WEIGHTS}/phase2_best.keras")

### Phase 2 — Evaluate on Test Set

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"\n📊 Phase 2 — Test Accuracy: {test_acc:.4f}")
print(f"📊 Phase 2 — Test Loss:     {test_loss:.4f}")

### Phase 2 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_phase2.history['accuracy'], label='Train Acc')
axes[0].plot(history_phase2.history['val_accuracy'], label='Val Acc')
axes[0].set_title('Phase 2 — Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_phase2.history['loss'], label='Train Loss')
axes[1].plot(history_phase2.history['val_loss'], label='Val Loss')
axes[1].set_title('Phase 2 — Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('MobileNetV2 — Phase 2: Fine-Tuning', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---

## 📚 Summary

| Metric | Phase 1 (Transfer Learning) | Phase 2 (Fine-Tuning) |
|--------|:---:|:---:|
| Base model | MobileNetV2 (frozen) | MobileNetV2 (top layers unfrozen) |
| Trainable params | Head only | Head + top backbone layers |
| Learning rate | Default Adam | 1e-5 |
| Model saved to | `weights/mobilenet/transferlearning/` | `weights/mobilenet/finetune/` |

### Key Takeaways for Students

1. **Transfer learning is powerful** — pre-trained features give a huge head start.
2. **Two-phase training matters** — train the head first, then fine-tune the backbone.
3. **Use a very low learning rate for fine-tuning** — large updates would destroy pre-trained features.
4. **Freeze early layers** — they capture universal features that don't need adjustment.
5. **Always re-compile after changing `.trainable`** — the optimizer must be rebuilt.